[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/06_%E3%83%99%E3%82%A4%E3%82%BA%E7%B5%B1%E8%A8%88_%E4%BA%8B%E5%BE%8C%E5%88%86%E5%B8%83%E3%81%A8%E4%BF%A1%E7%94%A8%E5%8C%BA%E9%96%93.ipynb)

In [ ]:
# === ① セットアップ（最初に実行してください）===
import pandas as pd               # 表データ
import numpy as np                # 数値計算
import matplotlib.pyplot as plt   # グラフ描画
import os
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の文字化け防止
# ローカルに data/ が無ければ公開リポジトリ(raw)から読み込む
RAW = 'https://raw.githubusercontent.com/Carlo-Broschi/statistics-python-for-students/main/data/'
def load(name):
    p = f'../data/{name}'
    return pd.read_csv(p) if os.path.exists(p) else pd.read_csv(RAW + name)
print('準備OK')
from scipy import stats

# 発展マーケ-06. ベイズ統計（事後分布・信用区間）

> 🟢 Colab（ブラウザ）で実行できます。**最初に「① セットアップ」セルを実行**してください。
> 📌 **発展編（統計検定2級の先：準1級〜実務／データサイエンス）**。statsmodels・scipy を使います（Colabは導入済み。ローカルは `uv add statsmodels scipy`）。

**舞台設定**：A/Bテストで「Bの方が良さそう」。でも頻度論のp値は分かりにくい。ベイズなら **「BがAより良い確率は92%」** のように、知りたい形で答えが出ます。

**使う統計（読む=準1級／本質=1級）**：事前分布・事後分布・ベータ二項共役・信用区間。

### 📋 使うデータ：`ab_test.csv`（LPボタン色のA/Bテスト）
1行＝訪問1件。`申込`は 1=申込/0=非申込。グループは A_青ボタン / B_緑ボタン。

## 🎯 この章でできるようになること
- 頻度論とベイズの考え方の違い（パラメータを「分布」で扱う）を説明できる
- ベータ二項の共役で **事後分布** を求められる
- **信用区間** と **P(B>A)** で意思決定できる

> **前提**：確率分布・A/Bテスト（実践03）　/　**所要**：約30分　/　**レベル**：発展（読む準1級・本質1級）

## 1. 申込率を「点」ではなく「分布」で持つ

頻度論は申込率を1つの値で推定します。ベイズは申込率そのものを**分布**で表し、データを見るほど分布が狭く＝確信が強くなります。`事後 ∝ 事前 × 尤度` が基本式です。

In [ ]:
ab = load('ab_test.csv')
g = ab.groupby('グループ')['申込'].agg(['sum','count'])
g.columns = ['申込数','訪問数']
print(g)

## 2. ベータ二項の共役：事後分布を求める

申込/非申込のような0/1データでは、事前に **ベータ分布 Beta(1,1)＝一様**（まっさらな知識）を置くと、事後は **Beta(1+申込数, 1+非申込数)** になります（共役なので計算一発）。

In [ ]:
post = {}
for grp, row in g.iterrows():
    a = 1 + row['申込数']; b = 1 + (row['訪問数'] - row['申込数'])   # 事後 Beta(a,b)
    post[grp] = stats.beta(a, b)
x = np.linspace(0.05, 0.16, 400)
fig, ax = plt.subplots(figsize=(7,4.5))
for grp, d in post.items():
    ax.plot(x, d.pdf(x), label=grp)
ax.set_xlabel('申込率'); ax.set_ylabel('事後分布の密度'); ax.legend()
ax.set_title('各グループの申込率の事後分布'); plt.show()

## 3. 信用区間（credible interval）

事後分布の中央95%が **95%信用区間**。頻度論の信頼区間と違い、「真の申込率がこの区間に入る確率が95%」と**そのまま**言えます。

In [ ]:
for grp, d in post.items():
    lo, hi = d.ppf([0.025, 0.975])
    print(f'{grp}: 事後平均 {d.mean():.3%}  95%信用区間 [{lo:.3%}, {hi:.3%}]')

## 4. P(B>A)：Bが勝つ確率と意思決定

2つの事後分布から乱数を大量に発生させ、「BのほうがAより高い」割合を数えれば、**BがAより良い確率**が出ます。これが意思決定にそのまま使えます。

In [ ]:
sa = post['A_青ボタン'].rvs(100000, random_state=1)
sb = post['B_緑ボタン'].rvs(100000, random_state=2)
p_b_better = (sb > sa).mean()
expected_lift = (sb - sa).mean()
print(f'P(B > A) = {p_b_better:.1%}')
print(f'Bにしたときのリフトの期待値 = {expected_lift:+.3%}ポイント')
print('→ 確率と期待リフトの両方を見て「Bに切り替えるか」を判断できる')

## 🧭 まとめ
- ベイズはパラメータを **分布** で扱い、`事後 ∝ 事前 × 尤度` で更新する。
- 0/1データは **ベータ二項の共役** で事後が一発（Beta(1+成功, 1+失敗)）。
- **信用区間** と **P(B>A)** は、p値より意思決定に直結する。

> ⚠️ **よくある間違い**
> - 信用区間と信頼区間は**解釈が別物**。信用区間は「真値がこの中に95%」と言ってよい。
> - データが少ないと **事前分布の影響**が残る（事前の選び方に注意）。データが増えれば頻度論と一致していく。
> - P(B>A)が高くても差がごく小さいことがある。**期待リフト**も併せて見る。

## 🧠 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます。

In [ ]:
# 採点用の関数（答え合わせに使うだけ・答えは表示しません）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
# Q1: 申込8・非申込2。事前Beta(1,1)のとき事後Beta(a,b)の a=1+申込 を ans に
ans = None   # 例: 1+8
_check('Q1 事後のa', ans, 1+8)

In [ ]:
# Q2: 事後がBeta(a=9, b=3)のとき事後平均 a/(a+b) を ans に
ans = None   # 例: 9/(9+3)
_check('Q2 事後平均', ans, 9/(9+3))

In [ ]:
# Q3: 事後がBeta(a=2, b=8)のとき事後平均 a/(a+b) を ans に
ans = None   # 例: 2/(2+8)
_check('Q3 事後平均(少数データ)', ans, 2/(2+8))

---
## 🏆 練習問題

**問1.** 事前を `Beta(1,1)` から `Beta(10,90)`（事前に申込率10%と強めに信じる）に変えると、少数データのときに事後がどう変わるか確かめよう。

**問2.** `P(B>A)` を、サンプリングではなく各群の事後平均だけで比べた場合と結果が一致するか見よう。

**問3.**（考察）「BがAより1%ポイント以上良い確率」を計算するには、`sb - sa` をどう使えばよい？

> 🔑 **解答例は別ページ**（ネタバレ防止）👉 **[解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/06_%E3%83%99%E3%82%A4%E3%82%BA%E7%B5%B1%E8%A8%88_%E4%BA%8B%E5%BE%8C%E5%88%86%E5%B8%83%E3%81%A8%E4%BF%A1%E7%94%A8%E5%8C%BA%E9%96%93.md)**
>
> 自分で解いてから開きましょう。

## 📒 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 事前分布 | データを見る前の信念 |
| 尤度 | データの出やすさ |
| 事後分布 | 事前×尤度。更新後の信念 |
| 共役 | 事前と事後が同じ分布族（計算が楽） |
| 信用区間 | 真値がこの中に入る確率◯% |

```python
from scipy import stats
post = stats.beta(1+成功, 1+失敗)         # ベータ二項の事後
post.mean(); post.ppf([0.025, 0.975])     # 事後平均・信用区間
(post_B.rvs(10**5) > post_A.rvs(10**5)).mean()   # P(B>A)
```